# MP2: Czyszczenie danych i feature engineering

**MajsterPlus — Predykcja rezygnacji klientów**

W tym mini-projekcie przejdziesz przez **obowiązkowy 12-krokowy pipeline przetwarzania danych**,
aby oczyścić dane i przygotować je do modelowania. Kroki muszą być wykonywane w podanej kolejności.

**Faza CRISP-DM**: Przygotowanie danych

---

## 0. Konfiguracja i reprodukowalność

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Weryfikacja wersji bibliotek
import sklearn, pandas as pd
assert sklearn.__version__.startswith(("1.4", "1.5")), (
    f"scikit-learn {sklearn.__version__} — expected 1.4.x or 1.5.x"
)
assert pd.__version__.startswith("2."), (
    f"pandas {pd.__version__} — expected 2.x"
)
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} — expected <2.0"
)
print(f"sklearn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")
print(f"Random seed: {SEED}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## Krok 1–2: Wczytanie danych i weryfikacja sumy kontrolnej

In [ ]:
import hashlib
from pathlib import Path

# Wykrywanie środowiska Colab
try:
    import google.colab
    DATA_DIR = Path("/content/drive/MyDrive/PUM/2. data")
except ImportError:
    DATA_DIR = Path("../2. data")

assert DATA_DIR.exists(), f"Nie znaleziono katalogu z danymi: {DATA_DIR}"

# Wczytanie i weryfikacja customers.csv
customers = pd.read_csv(DATA_DIR / "customers.csv")
cust_md5 = hashlib.md5((DATA_DIR / "customers.csv").read_bytes()).hexdigest()
assert cust_md5 == "c016cd4bfc36ac8d0a99bb6a3d55fa44", (
    f"customers.csv niezgodność MD5: {cust_md5} != c016cd4bfc36ac8d0a99bb6a3d55fa44"
)
print(f"customers.csv wczytano: {customers.shape}, MD5 OK")

# Wczytanie i weryfikacja transactions.csv
txn_md5 = hashlib.md5((DATA_DIR / "transactions.csv").read_bytes()).hexdigest()
assert txn_md5 == "a9c4bcfc4878baae6f5c250d4d15d737", (
    f"transactions.csv niezgodność MD5: {txn_md5} != a9c4bcfc4878baae6f5c250d4d15d737"
)
print(f"transactions.csv zweryfikowano: MD5 OK")

## Krok 3: Wyodrębnienie zmiennej docelowej i usunięcie ID

In [ ]:
# Zapisanie zmiennej docelowej przed jakimikolwiek transformacjami
y = customers["is_lapsed"].copy()
print(f"Zmienna docelowa zapisana: {y.shape[0]} wartości, odsetek rezygnacji = {y.mean():.3f}")

# Usunięcie customer_id (nie jest cechą) oraz is_lapsed (zmienna docelowa)
# Zachowanie gender do późniejszej analizy sprawiedliwości
gender_series = customers["gender"].copy()

df = customers.drop(columns=["customer_id", "is_lapsed"])
print(f"Roboczy DataFrame: {df.shape}")

## Krok 4: Parsowanie polskich dat

Kolumna `registration_date` zawiera polskie skróty miesięcy (np. "15-sty-2024").
Udostępniamy funkcję pomocniczą do parsowania — Twoim zadaniem jest ją zastosować i zweryfikować wynikową cechę.

In [ ]:
# Funkcja pomocnicza: parsowanie polskich dat na obiekty datetime
POLISH_MONTHS = {
    "sty": 1, "lut": 2, "mar": 3, "kwi": 4, "maj": 5, "cze": 6,
    "lip": 7, "sie": 8, "wrz": 9, "paź": 10, "lis": 11, "gru": 12,
}

def parse_polish_date(date_str: str) -> pd.Timestamp:
    """Konwertuje polski ciąg daty 'DD-mmm-YYYY' na Timestamp."""
    day, month_str, year = date_str.split("-")
    return pd.Timestamp(year=int(year), month=POLISH_MONTHS[month_str], day=int(day))

In [ ]:
# TODO: Zastosuj funkcję pomocniczą parse_polish_date do kolumny registration_date.
# Użyj daty referencyjnej 2025-01-15, aby wyliczyć wiek konta w dniach.
# Zweryfikuj, czy wyliczone wartości zgadzają się z istniejącą kolumną account_age_days.
# Następnie usuń registration_date (jest teraz redundantna).

# TWÓJ KOD TUTAJ


## Krok 5: Czyszczenie total_spend

Kolumna `total_spend` zawiera ciągi w formacie walutowym, np. `"PLN 1,234.50"`.
Przekonwertuj ją na kolumnę numeryczną typu float.

In [ ]:
# TODO: Przekonwertuj total_spend z formatu walutowego ("PLN 1,234.50") na liczbę typu float.
# Po konwersji wyświetl df["total_spend"].describe() w celu weryfikacji.

# TWÓJ KOD TUTAJ


### Interpretacja: rozkład total_spend

**TODO**: Porównaj średnią i medianę `total_spend` (użyj wyników `.describe()` powyżej). Co rozbieżność między tymi dwiema miarami mówi o kształcie rozkładu? Czy jest prawoskośny, lewoskośny, czy w przybliżeniu normalny? Dlaczego może to mieć znaczenie dla modelowania?

In [ ]:
# TODO: Wyświetl średnią i medianę total_spend.
# Napisz swoją interpretację jako komentarz lub instrukcję print poniżej.

# TWÓJ KOD TUTAJ


## Krok 6: Obsługa niemożliwych wartości satisfaction_score

Prawidłowy zakres dla `satisfaction_score` to [1.0, 5.0]. Wartości poza tym zakresem
to błędy wprowadzania danych i należy je zastąpić wartością NaN.

### Zastanów się

Dlaczego niemożliwe wartości muszą być zastąpione NaN **przed** uruchomieniem imputation w Kroku 7? Co by się stało z medianą, gdyby niemożliwe wartości (np. 0.0, 7.2, -1.0) zostały uwzględnione w obliczeniach?

In [ ]:
# TODO: Zidentyfikuj wartości satisfaction_score poza prawidłowym zakresem [1.0, 5.0]
# i zastąp je wartością NaN. Wyświetl liczbę znalezionych niemożliwych wartości.

# TWÓJ KOD TUTAJ


## Krok 7: Uzupełnianie brakujących wartości (imputation)

Po krokach 4–6 kilka kolumn zawiera brakujące wartości. Zastosuj odpowiednią imputation:
- **Kolumny numeryczne** (age, online_ratio, satisfaction_score): imputation medianą
- **Kolumny kategoryczne** (monthly_income_bracket, referral_source): imputation modą

In [ ]:
# TODO: Uzupełnij wszystkie brakujące wartości.
# Dla kolumn numerycznych użyj mediany; dla kolumn kategorycznych użyj mody.
# Wyświetl liczbę brakujących wartości przed i po imputation.
# Pamiętaj: age został wczytany jako Int64 (nullable int); po imputation przekonwertuj na zwykły int.

# TWÓJ KOD TUTAJ


## Krok 8: Usuwanie wartości odstających metodą IQR

Kolumna `avg_basket_value` zawiera skrajne wartości odstające. Użyj metody IQR
ze współczynnikiem 1.5, aby je zidentyfikować i usunąć.

In [ ]:
# Obliczenie IQR (uzupełnione)
Q1 = df["avg_basket_value"].quantile(0.25)
Q3 = df["avg_basket_value"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f"Granice IQR: [{lower_bound:.2f}, {upper_bound:.2f}]")

In [ ]:
# TODO: Odfiltruj wiersze, w których avg_basket_value wykracza poza granice IQR.
# WAŻNE: Musisz również przefiltrować y i gender_series tą samą maską logiczną.
# Dlaczego? Zastanów się, co się stanie, gdy wiersze cech i wartości docelowe się rozjadą.
# Wyświetl liczbę usuniętych wierszy.

# TWÓJ KOD TUTAJ


## Krok 9: Kodowanie zmiennych kategorycznych

Zakoduj wszystkie kolumny kategoryczne w następującej kolejności:
1. **Binarna**: `loyalty_member` (Tak/Nie)
2. **Porządkowa**: `monthly_income_bracket` (od A do E, reprezentujące rosnący dochód)
3. **One-hot encoding**: pozostałe kolumny kategoryczne z `pd.get_dummies(drop_first=False)`
4. Posortuj kolumny alfabetycznie i przekonwertuj wszelkie kolumny boolowskie na int.

In [ ]:
# TODO: Zakoduj wszystkie zmienne kategoryczne zgodnie z powyższą kolejnością.
# Po kodowaniu posortuj kolumny alfabetycznie: df = df[sorted(df.columns)]
# Wyświetl df.shape i df.dtypes w celu weryfikacji.

# TWÓJ KOD TUTAJ


### Interpretacja: kompromis drop_first

**TODO**: W kursie używamy `drop_first=False`. Co by się stało, gdybyś użył `drop_first=True`? Jakie typy modeli skorzystałyby na usunięciu pierwszej kategorii i dlaczego? Jakie typy modeli nie byłyby tym dotknięte?

In [ ]:
# TODO: Napisz swoją odpowiedź jako komentarz lub instrukcję print.
# Rozważ: Czym jest wieloliniowość (multicollinearity)? Które modele są na nią wrażliwe?

# TWOJA ODPOWIEDŹ TUTAJ


## Krok 10: Bramka asercji null

In [ ]:
# To MUSI przejść przed kontynuowaniem
assert df.isnull().sum().sum() == 0, (
    f"Nadal masz {df.isnull().sum().sum()} wartości null! Popraw kroki 6-7."
)
print(f"Asercja null przeszła pomyślnie. Kształt DataFrame: {df.shape}")
print(f"Wszystkie typy numeryczne: {all(df.dtypes.apply(lambda x: np.issubdtype(x, np.number)))}")

## Krok 11: Podział na zbiór treningowy i testowy (train/test split)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42, stratify=y
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Odsetek rezygnacji y_train: {y_train.mean():.3f}")
print(f"Odsetek rezygnacji y_test:  {y_test.mean():.3f}")

## Krok 12: Skalowanie cech (StandardScaler)

Dopasuj StandardScaler na zbiorze treningowym i przekształć zarówno zbiór treningowy, jak i testowy.
Pamiętaj: dopasowanie tylko na zbiorze treningowym, transformacja obu -- dopasowanie na pełnych danych to data leakage.

In [ ]:
# TODO: Zastosuj StandardScaler. Dopasuj TYLKO na danych treningowych, następnie przekształć oba zbiory.
# Zachowaj strukturę DataFrame (kolumny i indeks) w przeskalowanym wyniku.

from sklearn.preprocessing import StandardScaler

# TWÓJ KOD TUTAJ


## Bonus: Grupowanie K-Means (clustering)

**Opcjonalnie**: Użyj K-Means, aby stworzyć segmenty klientów jako dodatkową cechę.
Użyj 3 klastrów na przeskalowanych cechach behawioralnych (purchase_count, avg_basket_value,
days_since_last_purchase, product_categories_bought).

**Zadanie**: Dopasuj K-Means, dodaj etykiety klastrów jako cechę do obu zbiorów (treningowego i testowego).

In [ ]:
# TWÓJ KOD TUTAJ (opcjonalny bonus)
# from sklearn.cluster import KMeans
# Dopasuj na cechach behawioralnych ze zbioru treningowego
# Dodaj etykiety klastrów do X_train_scaled i X_test_scaled

## Zapis checkpoint dla MP3

In [ ]:
import pickle

CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Podział gender_series zgodnie z podziałem train/test
gender_train = gender_series.loc[X_train.index]
gender_test = gender_series.loc[X_test.index]

checkpoint = {
    "X_train": X_train_scaled if "X_train_scaled" in dir() else X_train,
    "X_test": X_test_scaled if "X_test_scaled" in dir() else X_test,
    "y_train": y_train,
    "y_test": y_test,
    "feature_names": list(df.columns),
    "gender_train": gender_train,
    "gender_test": gender_test,
}

with open(CHECKPOINT_DIR / "mp2_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)
print(f"Checkpoint zapisany: {CHECKPOINT_DIR / 'mp2_checkpoint.pkl'}")
print(f"Klucze: {list(checkpoint.keys())}")

---
*Koniec MP2 — Przejdź do MP3: Modelowanie baseline i porównanie algorytmów*